# Notebook 03 — RAG Pipeline Observability & Evaluation

This notebook observes the **Phase 3 augmented generation pipeline** end-to-end.

What we will inspect:
1. Browse test questions from the HuggingFace evaluation set
2. Run a single RAG query and trace every step (retrieved passages → prompt → answer)
3. Inspect the augmented prompt sent to GPT-4o
4. Run a small evaluation batch and display accuracy metrics

**Prerequisite:** the API server must be running and data must be ingested.
```
uv run uvicorn main:app --app-dir src --port 8004
# POST http://localhost:8004/ingest?limit=50
```

In [1]:
import json
import requests
import pandas as pd
from IPython.display import display, Markdown

BASE_URL = "http://localhost:8004"

resp = requests.get(f"{BASE_URL}/health")
print("Server:", resp.json()["status"])

Server: ok


## 1. Browse Test Questions

The HuggingFace `test.parquet` contains pre-built Q&A pairs for evaluation.
These are the questions we use to measure how well the RAG system performs.

In [2]:
resp = requests.get(f"{BASE_URL}/test-questions?limit=10")
body = resp.json()

print(f"Total test questions: {body['total']}\n")
rows = [{"id": q["id"], "question": q["question"], "expected_answer": q["answer"]}
        for q in body["questions"]]
display(pd.DataFrame(rows))

Total test questions: 918



,id,question,expected_answer
0,0,Was Abraham Lincoln the sixteenth President of...,yes
1,2,Did Lincoln sign the National Banking Act of 1...,yes
2,4,Did his mother die of pneumonia?,no
3,6,How many long was Lincoln's formal education?,18 months
4,8,When did Lincoln begin his political career?,1832
5,10,What did The Legal Tender Act of 1862 establish?,"the United States Note, the first paper curren..."
6,12,Who suggested Lincoln grow a beard?,11-year-old Grace Bedell
7,14,When did the Gettysburg address argue that Ame...,1776
8,16,Did Lincoln beat John C. Breckinridge in the 1...,yes
9,18,Was Abraham Lincoln the first President of the...,No


## 2. Run a Single RAG Query — Full Trace

`POST /query` returns the answer **and** every intermediate artefact:
- `retrieved` — the top-K passages with similarity scores  
- `prompt` — the exact augmented prompt sent to GPT-4o  
- `answer` — the generated answer

In [3]:
question = "What is Uruguay?"
resp = requests.post(f"{BASE_URL}/query", params={"question": question, "top_k": 3})
result = resp.json()

print(f"Question : {result['question']}")
print(f"Model    : {result['model']}")
print(f"\n{'='*60}")
print(f"ANSWER:\n{result['answer']}")

Question : What is Uruguay?
Model    : gpt-4o

ANSWER:
Uruguay is a country located in the southeastern part of South America. It is officially known as the Eastern Republic of Uruguay. The country is home to 3.3 million people, with 1.7 million residing in the capital, Montevideo, and its metropolitan area. Uruguay is the second smallest sovereign nation in South America, after Suriname, and the third smallest territory, with French Guiana being the smallest. The landscape is characterized by rolling plains, low hill ranges, and a fertile coastal lowland, with a dense network of rivers and several lagoons along the Atlantic coast. Uruguay has a middle-income economy dominated by the state services sector, an export-oriented agricultural sector, and an industrial sector. The country is heavily reliant on trade, particularly in agricultural exports, and has developed a reputation for stable financial indicators and investment-grade sovereign bond ratings. In recent years, Uruguay has al

In [4]:
# Inspect the retrieved passages that fed the answer
print("Retrieved passages:\n")
rows = [
    {"rank": r["rank"], "score": round(r["score"], 4), "passage_id": r["passage_id"],
     "text_preview": r["text"][:90] + "..."}
    for r in result["retrieved"]
]
display(pd.DataFrame(rows))

Retrieved passages:



,rank,score,passage_id,text_preview
0,1,0.6151,0,"Uruguay (official full name in ; pron. , Eas..."
1,2,0.5710,25,"At 176,214 square kilometres (68,036 square mi..."
2,3,0.5658,37,"Uruguay has a middle income economy, mainly do..."


## 3. Inspect the Augmented Prompt

This is the exact prompt that was sent to GPT-4o — the retrieved context injected alongside the user's question.

In [5]:
print("AUGMENTED PROMPT SENT TO GPT-4o:")
print("=" * 60)
print(result["prompt"])

AUGMENTED PROMPT SENT TO GPT-4o:
Use the following context passages to answer the question.

Context:
[1] Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.

[2] At 176,214 square kilometres (68,036 square miles) of continental land and 142,199 square kilometres (54,903 sq mi) of jurisdictional waters and small river islands,  Instituto Nacional Estadistica  Uruguay is the second smallest sovereign nation in South America (after Suriname) and the third smallest territory (French Guiana is the smallest). The landscape features mostly rolling plains and low hill ranges (cuchillas) with a fertile coastal lowland. A dense fluvial network covers the country, consisting of four river basins or deltas; the RÃ­o de la Plata, the Uruguay River, the Laguna MerÃ­n and the RÃ­o Negro. The major intern

## 4. Evaluation — Batch Scoring

Run a set of test questions through the full pipeline and score them using substring-match accuracy.
Each result shows the question, expected answer, generated answer, and whether it was correct.

In [6]:
# Fetch the first 5 test questions for evaluation
test_resp = requests.get(f"{BASE_URL}/test-questions?limit=5")
test_qs = test_resp.json()["questions"]

eval_rows = []
for tq in test_qs:
    r = requests.post(
        f"{BASE_URL}/query",
        params={"question": tq["question"], "top_k": 5},
    )
    body = r.json()
    generated = body.get("answer", "")
    expected = tq["answer"]
    correct = expected.lower() in generated.lower()
    eval_rows.append({
        "question": tq["question"],
        "expected": expected,
        "generated": generated[:120] + ("..." if len(generated) > 120 else ""),
        "correct": "✅" if correct else "❌",
    })

df_eval = pd.DataFrame(eval_rows)
correct_count = sum(1 for r in eval_rows if r["correct"] == "✅")
print(f"Accuracy: {correct_count}/{len(eval_rows)} ({correct_count/len(eval_rows)*100:.0f}%)\n")
display(df_eval)

Accuracy: 3/5 (60%)



,question,expected,generated,correct
0,Was Abraham Lincoln the sixteenth President of...,yes,"Yes, Abraham Lincoln was the sixteenth Preside...",✅
1,Did Lincoln sign the National Banking Act of 1...,yes,"Yes, President Abraham Lincoln signed the Nati...",✅
2,Did his mother die of pneumonia?,no,The provided context passages do not contain a...,✅
3,How many long was Lincoln's formal education?,18 months,The provided context passages do not contain a...,❌
4,When did Lincoln begin his political career?,1832,The provided context passages do not contain a...,❌
